# Calculations of cooling tower with Poppe method

## Import of libraries

In [5]:
from lib import AirFlow, WaterFlow, PoppeSolver, u, Q_, MerkelSolver
air = AirFlow(temp=Q_(20, u.degC), humidity=Q_(60, u.perc))
water = WaterFlow(temp=Q_(45, u.degC), salinity=Q_(1500, u.mg/u.kg))
solver = PoppeSolver(air, water, water_temp_out=Q_(28, u.degC), lg_ratio=1.2)
me, df = solver.solve_with_merkel_number()
print("Poppe results:")
print(df)
# Evaporation
evap_loss = df['air_omega_kg_kg'].iloc[-1] - df['air_omega_kg_kg'].iloc[0]
evap_loss /= 1.2
print(f"\nИспарение: {evap_loss * 1000:.2f} г воды на 1 кг воды")

Poppe results:
    water_temp_c  air_temp_c  air_rh_perc  air_omega_kg_kg  air_enthalpy_j_kg  \
0      28.000000   20.000000    60.000000         0.008734       42289.860053   
1      28.346939   20.277310    62.798269         0.009309       44030.672217   
2      28.693878   20.557267    65.460408         0.009881       45771.503158   
3      29.040816   20.839635    67.990967         0.010453       47512.352877   
4      29.387755   21.124178    70.394581         0.011023       49253.221372   
5      29.734694   21.410662    72.675944         0.011593       50994.108645   
6      30.081633   21.698853    74.839784         0.012161       52735.014695   
7      30.428571   21.988522    76.890842         0.012729       54475.939522   
8      30.775510   22.279443    78.833848         0.013296       56216.883127   
9      31.122449   22.571392    80.673503         0.013862       57957.845508   
10     31.469388   22.864152    82.414463         0.014428       59698.826667   
11     31.816

In [6]:
evap_loss = df['air_omega_kg_kg'].iloc[-1] - df['air_omega_kg_kg'].iloc[0]
evap_loss /= 1.2

# Правильно нужно разделить на пар и туман:
omega_out = df['air_omega_kg_kg'].iloc[-1]
omega_in  = df['air_omega_kg_kg'].iloc[0]
fog_out   = df['fog_kg_kg'].iloc[-1]  # это уже кг/кг несмотря на название колонки!

# Только пар (без тумана):
evap_vapor = (omega_out - fog_out - omega_in) / 1.2

# Туман (улетает с воздухом):
evap_fog = fog_out / 1.2

# Полные потери воды:
evap_total = (omega_out - omega_in) / 1.2

print(f"Испарение (пар):  {evap_vapor*1000:.3f} г/кг воды")
print(f"Унос тумана:      {evap_fog*1000:.4f} г/кг воды")
print(f"Итого потери E:   {evap_total*1000:.3f} г/кг воды")

Испарение (пар):  22.805 г/кг воды
Унос тумана:      0.0014 г/кг воды
Итого потери E:   22.806 г/кг воды


In [7]:
print(me)

1.2585674847449013


In [11]:
1.2587/(1.2**-0.6)

1.4042065025713337

In [15]:
merkel_solver = MerkelSolver(1.9, 0.6, air, water, Q_(28, u.degC))

In [16]:
merkel_solver.solve_me(1.2)

1.258727750135329

In [22]:
float((df["zone"] == "fog").sum()/df["zone"].size*100.0)

48.0